<a href="https://colab.research.google.com/github/Irfan-code-cloud/ML-Internship-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Retrieve Hugging Face token from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    print("✅ Found HF_TOKEN in secrets.")
except Exception as e:
    raise ValueError("❌ Please store your Hugging Face read token in Colab Secrets named 'HF_TOKEN'.")

# 2. Connect DuckDB & setup Hugging Face authentication
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Base path for FlyRank Hugging Face Parquet dataset
BASE_DATA_PATH = "hf://datasets/FlyRank/internship-warehouse"

# Quick connectivity test on daily performance table
row_count = con.sql(f"SELECT COUNT(*) FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/**/*.parquet')").fetchone()[0]
print(f"✅ Successfully connected to FlyRank Warehouse dataset! Total rows in fact table: {row_count:,}")

✅ Found HF_TOKEN in secrets.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Successfully connected to FlyRank Warehouse dataset! Total rows in fact table: 78,835,655


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis & Time Window Statement

* **One row =** One unique combination of `client_hash_id` + `content_hash_id` + `report_date` (in `fact_content_daily_performance`).
* **Over which dates =** The mid-panel observation month from **March 1, 2026 (`2026-03-01`) to March 31, 2026 (`2026-03-31`)** (`month=2026-03`).

#### Empirical Verification Results:
* **Date Range:** 2026-03-01 to 2026-03-31
* **Total Rows:** 9,841,378
* **Unique Grain Count:** 9,841,378
* **Uniqueness Status:** PASSED (Zero duplicate rows across client + content + date).

In [5]:
# =====================================================================
# SECTION 1 VERIFICATION: GRAIN AND DATE RANGE IN WAREHOUSE (FIXED SCHEMA)
# =====================================================================

# First, let's inspect the actual column names in the parquet files
schema_df = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

# Detect the date column dynamically (usually 'date' or 'report_date')
date_col = [c for c in schema_df['column_name'].tolist() if 'date' in c.lower()]
date_col_name = date_col[0] if date_col else 'date'

verification_query = f"""
SELECT
    MIN({date_col_name}) AS start_date,
    MAX({date_col_name}) AS end_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST({date_col_name} AS VARCHAR))) AS unique_grain_count
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

res = con.sql(verification_query).df()
display(res)

# Print explicit proof statement
is_unique = res['total_rows'][0] == res['unique_grain_count'][0]
print(f"\n✅ Date Range: From {res['start_date'][0]} to {res['end_date'][0]}")
print(f"✅ Total Rows: {res['total_rows'][0]:,}")
print(f"✅ Grain Uniqueness Check: {'PASSED (One row = unique client + content + date)' if is_unique else 'FAILED'}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows,unique_grain_count
0,2026-03-01,2026-03-31,9841378,9841378



✅ Date Range: From 2026-03-01 00:00:00 to 2026-03-31 00:00:00
✅ Total Rows: 9,841,378
✅ Grain Uniqueness Check: PASSED (One row = unique client + content + date)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: Feature / Label / Context / Excluded

Every field in the dataset that is accessed or evaluated is categorized into one of four functional buckets below:

---

### 1. 🎯 Label Bucket (Target & Proxy Metrics)
Fields used to construct ground-truth targets or calculate business optimization metrics:
* **`gsc_clicks` / `gsc_impressions`**: Historical performance metrics used to derive actual observed Click-Through Rate ($\text{CTR} = \frac{\text{clicks}}{\text{impressions}}$).
* **`expected_ctr`**: Continuous regression target predicted by the ML model based on features.
* **`ctr_gap`**: Derived business proxy label computed via rule: $\text{Predicted Expected CTR} - \text{Actual Observed CTR}$.

---

### 2. ⚡ Feature Bucket (Model Predictors)
Input fields used by the model to predict expected engagement and CTR potential:
* **`gsc_position`**: Search engine result page (SERP) average position (strong non-linear relationship with CTR).
* **`ga4_total_engagement_sec`**: On-page user engagement duration.
* **`ai_gemini` / `ai_meta` / `ai_other`**: Indicators/scores for AI-assisted content optimization or generation.
* **`gsc_data_available` / `ga4_data_available`**: Boolean flags indicating data completeness for tracking integration.

---

### 3. 🏷️ Context Bucket (Identifiers & Partitioning)
Primary keys, foreign keys, and temporal boundaries used to structure the grain and aggregate data:
* **`client_hash_id`**: Unique identifier for the client account.
* **`content_hash_id`**: Unique identifier for the specific content/URL piece.
* **`report_date`**: Daily temporal timestamp defining the primary observation grain.
* **`month`**: Partition key (e.g., `month=2026-03`) used for temporal scoping and directory filtering.

---

### 4. 🚫 Excluded Bucket (With Justifications)
Fields omitted from model training to prevent data leakage, redundancy, or poor signal quality:

| Field Name | Reason for Exclusion |
| :--- | :--- |
| **`client_has_gsc` / `client_has_ga4`** | **Redundant with row-level availability flags:** High-level client toggles duplicate the row-level `gsc_data_available` and `ga4_data_available` signals without adding predictive value. |
| **`June 2026 Data (`month=2026-06`)`** | **Data Leakage / Sealed Outcome Window:** June 2026 represents the final future test month. Excluded from feature engineering and training pipelines to avoid lookahead bias. |
| **Raw Unaggregated Text Identifiers** | **High Cardinality / Non-Predictive:** Raw hash strings (`client_hash_id`, `content_hash_id`) serve purely as joins/groupings in Context, not direct input features for statistical prediction. |

In [6]:
# =====================================================================
# SECTION 2: FIELD BUCKET CLASSIFICATION & DICTIONARY REGISTRATION
# =====================================================================

field_buckets = {
    "label": [
        "gsc_clicks",
        "gsc_impressions",
        "expected_ctr",
        "ctr_gap"
    ],
    "feature": [
        "gsc_position",
        "ga4_total_engagement_sec",
        "ai_gemini",
        "ai_meta",
        "ai_other",
        "gsc_data_available",
        "ga4_data_available"
    ],
    "context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "month"
    ],
    "excluded": {
        "client_has_gsc": "Redundant with row-level gsc_data_available flag",
        "client_has_ga4": "Redundant with row-level ga4_data_available flag",
        "month=2026-06": "Sealed future test month; excluded to prevent data leakage"
    }
}

# Verify bucket counts
print("✅ Field Buckets successfully registered:")
for category, fields in field_buckets.items():
    if isinstance(fields, list):
        print(f"  • {category.upper():<10} ({len(fields)} fields): {', '.join(fields)}")
    else:
        print(f"  • {category.upper():<10} ({len(fields)} rules/fields):")
        for f, reason in fields.items():
            print(f"      - {f}: {reason}")

✅ Field Buckets successfully registered:
  • LABEL      (4 fields): gsc_clicks, gsc_impressions, expected_ctr, ctr_gap
  • FEATURE    (7 fields): gsc_position, ga4_total_engagement_sec, ai_gemini, ai_meta, ai_other, gsc_data_available, ga4_data_available
  • CONTEXT    (4 fields): client_hash_id, content_hash_id, report_date, month
  • EXCLUDED   (3 rules/fields):
      - client_has_gsc: Redundant with row-level gsc_data_available flag
      - client_has_ga4: Redundant with row-level ga4_data_available flag
      - month=2026-06: Sealed future test month; excluded to prevent data leakage


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1: Grain Uniqueness Check
**Claim:** One row in our slice represents exactly one unique combination of `client_hash_id` + `content_hash_id` + `report_date`.

In [7]:
# =====================================================================
# QUERY 1: VERIFY GRAIN UNIQUENESS
# =====================================================================

q1_grain_check = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS unique_grain_count,
    COUNT(*) - COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS duplicate_count
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

res_q1 = con.sql(q1_grain_check).df()
display(res_q1)

print(f"✅ Claim Verified: Total rows ({res_q1['total_rows'][0]:,}) matches unique grain count ({res_q1['unique_grain_count'][0]:,}). Zero duplicates.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_count,duplicate_count
0,9841378,9841378,0


✅ Claim Verified: Total rows (9,841,378) matches unique grain count (9,841,378). Zero duplicates.


### Query 2: Slice Row Count and Date Span
**Claim:** The mid-panel slice for `month=2026-03` spans precisely from `2026-03-01` to `2026-03-31` with complete daily coverage.

In [8]:
# =====================================================================
# QUERY 2: VERIFY DATE SPAN AND ROW COUNT
# =====================================================================

q2_span_check = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_days,
    COUNT(*) AS total_slice_rows
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

res_q2 = con.sql(q2_span_check).df()
display(res_q2)

print(f"✅ Claim Verified: Date span covers {res_q2['distinct_days'][0]} days from {res_q2['min_date'][0]} to {res_q2['max_date'][0]} across {res_q2['total_slice_rows'][0]:,} rows.")

,min_date,max_date,distinct_days,total_slice_rows
0,2026-03-01,2026-03-31,31,9841378


✅ Claim Verified: Date span covers 31 days from 2026-03-01 00:00:00 to 2026-03-31 00:00:00 across 9,841,378 rows.


### Query 3: Availability & Missing Values Check
**Claim:** Filtering on valid data availability (`gsc_data_available IS TRUE` and `ga4_data_available IS TRUE`) removes non-integrated tracking rows and returns surviving usable records.

In [14]:
# =====================================================================
# QUERY 3: DUAL-SOURCE AVAILABILITY & DATA ATTRITION METRICS
# =====================================================================

q3_availability_check = f"""
SELECT
    COUNT(*) AS total_raw_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_available_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) AS fully_surviving_rows,
    -- Calculate explicit survival & attrition percentages
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_rate_pct,
    ROUND((1.0 - (COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) * 1.0 / COUNT(*))) * 100.0, 2) AS data_attrition_pct
FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet');
"""

res_q3 = con.sql(q3_availability_check).df()
display(res_q3)

# Extract scalar variables for explicit ML pipeline logging
total_raw = res_q3['total_raw_rows'][0]
surviving_rows = res_q3['fully_surviving_rows'][0]
survival_pct = res_q3['survival_rate_pct'][0]
attrition_pct = res_q3['data_attrition_pct'][0]

# Transparent Python Pipeline Output
print("\n----------------------------------------------------------------------")
print(f"⚠️ DATA ATTRITION ALERT (Preprocessing Filter Pass):")
print(f"  • Total Input Rows: {total_raw:,}")
print(f"  • Surviving Rows (GSC + GA4 Active): {surviving_rows:,} ({survival_pct}%)")
print(f"  • Attrition Loss Rate: {attrition_pct}%")
print("----------------------------------------------------------------------")

,total_raw_rows,gsc_available_rows,ga4_available_rows,fully_surviving_rows,survival_rate_pct,data_attrition_pct
0,9841378,3611061,413966,364347,3.7,96.3



----------------------------------------------------------------------
⚠️ DATA ATTRITION ALERT (Preprocessing Filter Pass):
  • Total Input Rows: 9,841,378
  • Surviving Rows (GSC + GA4 Active): 364,347 (3.7%)
  • Attrition Loss Rate: 96.3%
----------------------------------------------------------------------


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits & Blind Spots

While this slice establishes a strict baseline feature frame, structural tracking mechanics introduce inherent blind spots. **This data can NEVER tell us:**

* **True Historical Performance Prior to Linkage (Unbalanced History):**
  Because GA4 was integrated much later in many accounts' lifecycles, historical time-series features are severely skewed. A URL showing zero GA4 activity in early periods reflects a missing tracking tag or unlinked property, not a lack of real-world user engagement.

* **Early-Stage Conversions for GSC-Only Rows:**
  For the ~3.2M rows where `gsc_data_available IS TRUE` but GA4 is unlinked (`NULL`/`FALSE`), we can observe search intent (clicks, impressions, position) but are entirely blind to downstream user behavior. The data cannot inform us whether search visibility translated into actual site sessions, bounce rates, or target conversions.

* **Clean Causal Impact Across Window Overlaps:**
  Daily syncs from BigQuery and dynamic reporting windows introduce timestamp alignment lags between GSC search performance and GA4 session attribution. The dataset cannot prove direct intra-day causality between a specific search impression and a subsequent GA4 conversion event.

* **Generalizable Behavior Across Un-instrumented Accounts (Selection Bias):**
  Because requiring dual tracking (`gsc_data_available IS TRUE AND ga4_data_available IS TRUE`) drops 96.3% of raw volume (leaving 364,347 of 9.84M rows for `month=2026-03`), this slice reflects only high-intent, technically mature enterprise domains. It cannot predict behavior for long-tail or partially configured sites.

## Section 5: Feature Engineering & Temporal Availability

Below are 5 honest features extracted for the mid-panel month (`2026-03`). Each feature includes explicit justification for its availability at the decision moment.

### Feature Justifications
1. **`gsc_position_lag7`**: Knowable at the decision moment because it represents historical search ranking position logged prior to the prediction day.
2. **`gsc_impressions_lag7`**: Knowable at the decision moment because it measures past search visibility prior to the prediction day.
3. **`ga4_engagement_rate_lag7`**: Knowable at the decision moment because historical user interaction data from GA4 was already recorded in previous session logs.
4. **`url_path_depth`**: Knowable at the decision moment because the URL structure is fixed and deterministic at publication time.
5. **`is_weekend`**: Knowable at the decision moment because calendar dates are known in advance.

In [25]:
# Query 5 honest features using exact FlyRank schema columns

query = f"""
SELECT
    report_date AS date,
    content_hash_id AS content_id,

    -- Feature 1: Historical average ranking sum position (lagged 7d)
    AVG(gsc_sum_position) OVER (
        PARTITION BY content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS gsc_sum_position_lag7,

    -- Feature 2: Historical total impressions (lagged 7d)
    AVG(gsc_impressions) OVER (
        PARTITION BY content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS gsc_impressions_lag7,

    -- Feature 3: Historical scroll events / engagement activity (lagged 7d)
    AVG(scroll_events) OVER (
        PARTITION BY content_hash_id ORDER BY report_date ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS scroll_events_lag7,

    -- Feature 4: Day of month temporal feature
    DAYOFMONTH(report_date) AS day_of_month,

    -- Feature 5: Calendar temporal feature (weekend check)
    CASE WHEN DAYOFWEEK(report_date) IN (0, 6) THEN 1 ELSE 0 END AS is_weekend,

    -- Target variable
    gsc_clicks AS target_clicks

FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
"""

# Execute query using your active connection
df_features = con.sql(query).to_df().dropna().reset_index(drop=True)
print(f"✅ Feature Frame Shape: {df_features.shape}")
df_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Feature Frame Shape: (1958344, 8)


,date,content_id,gsc_sum_position_lag7,gsc_impressions_lag7,scroll_events_lag7,day_of_month,is_weekend,target_clicks
0,2026-03-07,content_001ddf6f43eeaf75,1239.500000,181.500000,0.0,7,1,1
1,2026-03-08,content_001ddf6f43eeaf75,1317.428571,180.857143,0.0,8,1,0
2,2026-03-09,content_001ddf6f43eeaf75,1360.000000,186.285714,0.0,9,0,0
3,2026-03-10,content_001ddf6f43eeaf75,1484.857143,192.857143,0.0,10,0,0
4,2026-03-11,content_001ddf6f43eeaf75,1470.857143,187.571429,0.0,11,0,1


In [23]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE_DATA_PATH}/fact_content_daily_performance/month=2026-03/*.parquet')").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## Section 6: Data Leakage Experiment

This section demonstrates the impact of data leakage by intentionally introducing a target-derived variable into the feature set and observing the artificial score spike.

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

# Define honest feature matrix matching df_features columns
honest_features = [
    'gsc_sum_position_lag7',
    'gsc_impressions_lag7',
    'scroll_events_lag7',
    'day_of_month',
    'is_weekend'
]

X_honest = df_features[honest_features]
y = df_features['target_clicks']

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# Fast, robust regressor suited for ~2M rows
model_honest = HistGradientBoostingRegressor(random_state=42)
model_honest.fit(X_train, y_train)

# Evaluate Honest Score
y_pred_honest = model_honest.predict(X_test)
honest_r2 = r2_score(y_test, y_pred_honest)

print(f"✅ Honest Baseline R² Score: {honest_r2:.4f}")

✅ Honest Baseline R² Score: 0.3288


In [27]:
# Create a leaked feature directly using target_clicks (same-day exact CTR calculation)
df_features['leaked_exact_ctr'] = df_features['target_clicks'] / (df_features['gsc_impressions_lag7'] + 1e-5)

# Build feature matrix including the leaked column
X_leaked = df_features[honest_features + ['leaked_exact_ctr']]

# Train / Test Split with Leaked Feature
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(X_leaked, y, test_size=0.2, random_state=42)

# Train Model on Leaked Data
model_leaked = HistGradientBoostingRegressor(random_state=42)
model_leaked.fit(X_tr_leak, y_tr_leak)

# Evaluate Leaked Score
y_pred_leaked = model_leaked.predict(X_te_leak)
leaked_r2 = r2_score(y_te_leak, y_pred_leaked)

print(f"⚠️ LEAKED Model R² Score: {leaked_r2:.4f} (Artificial Jump Near 1.0)")

⚠️ LEAKED Model R² Score: 0.9040 (Artificial Jump Near 1.0)


In [28]:
# Drop leaked feature and restore honest evaluation
X_reverted = X_leaked.drop(columns=['leaked_exact_ctr'])

X_tr_rev, X_te_rev, y_tr_rev, y_te_rev = train_test_split(X_reverted, y, test_size=0.2, random_state=42)

model_reverted = HistGradientBoostingRegressor(random_state=42)
model_reverted.fit(X_tr_rev, y_tr_rev)

y_pred_reverted = model_reverted.predict(X_te_rev)
reverted_r2 = r2_score(y_te_rev, y_pred_reverted)

print(f"🛡️ Reverted Honest R² Score: {reverted_r2:.4f}")
assert abs(reverted_r2 - honest_r2) < 1e-5, "Reverted score does not match honest baseline!"

🛡️ Reverted Honest R² Score: 0.3288


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.